In [1]:
import torch

import os
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_VERBOSE"] = "1" 

/home/u1501463/miniconda3/envs/sam_audio/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
from sam_audio import SAMAudio, SAMAudioProcessor
import torchaudio
import torch

model = SAMAudio.from_pretrained(
    "facebook/sam-audio-large",
    cache_dir='/home/u1501463/tool_use_LALM/models--facebook--sam-audio-large'
)
processor = SAMAudioProcessor.from_pretrained("facebook/sam-audio-large")
# model = model.eval().cuda()

model = model.eval().half().cuda()  # .half() 關鍵在這裡


/home/u1501463/miniconda3/envs/sam_audio/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/u1501463/miniconda3/envs/sam_audio/lib/python3.11/site-packages/imagebind/data.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 50901.75it/s]
/home/u1501463/miniconda3/envs/sam_audio/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/home/u1501463/miniconda3/envs/sam_audio/

In [3]:
model.text_ranker = model.text_ranker.half()
model.text_ranker.rankers = model.text_ranker.rankers.half()
# model.text_ranker.weights = model.text_ranker.weights.half()

In [17]:
file = 'example.wav'
description = "vocal"

In [18]:

batch = processor(
    audios=[file],
    descriptions=[description],
).to("cuda")

with torch.inference_mode():
    with torch.amp.autocast(dtype=torch.float16, device_type = 'cuda'):

    # NOTE: `predict_spans` and `reranking_candidates` have a large impact on performance.
    # Setting `predict_span=True` and `reranking_candidates=8` will give you better results at the cost of
    # latency and memory. See the "Span Prediction" section below for more details
        batch.audios = batch.audios.half()
        result = model.separate(batch, predict_spans=False)
        # result = model.separate(batch, predict_spans=True, reranking_candidates=8)
print('finish')


sample_rate = processor.audio_sampling_rate

target = result.target[0].cpu().float().unsqueeze(0)    # (1, time)
residual = result.residual[0].cpu().float().unsqueeze(0)  # (1, time)

torchaudio.save("target.wav", target, sample_rate)
torchaudio.save("residual.wav", residual, sample_rate)

print("Saved target.wav and residual.wav")

finish
Saved target.wav and residual.wav
